# 5.8 阶段六：QS 和 KVS 带尾块

此前所有阶段都假设 QS、KVS 是 128 的倍数——真实输入当然不会这么听话。阶段六解除这条约束：**QS 和 KVS 都允许带尾块**。

## 1. 尾块从哪来

回顾第 3 章 3.3 节：

- BSND/BSH 排布下 S 维 padding 对齐到最大值——补齐区是纯浪费；
- TND 排布天然变长——但按固定基本块切分必然产生尾块（如 456 = 3×128 + 72）。

无论哪种排布，Kernel 视角都要回答同一个问题：**最后一个 S1 块 / S2 块不足 128 行时怎么算？**

## 2. 尾块的三重代价

| 代价 | 说明 |
|--|--|
| 计算效率 | 矩阵 M 维不满，Cube 利用率下降 |
| 掩码处理 | 尾块行数不齐，行级 mask 边界更复杂（与 5.7 节叠加） |
| 流水收尾 | preload 流水的收尾段变长，断流比例上升 |

## 3. 处理方式：按实际行数计算

```cpp
// S2 尾块：本块实际行数 = s2Size - s2Idx * s2Block
int64_t curS2 = min(s2Block, s2Size - s2Idx * s2Block);

// 1) 搬运：只搬 curS2 行（不是整 128）
CopyInKV(s2Idx, curS2);

// 2) 矩阵乘：mm1/mm2 的 M/N 用 curS2（Matmul 高层 API 支持非整块 shape）
mm1.SetTensorA(q, false);
mm1.SetTensorB(k, true, curS2);     // B 的有效列 = curS2

// 3) softmax：有效行外填 0 / -inf，避免读未初始化数据
//    （V1 拍对 [curS2, s2Block) 的列做元素级掩码）

// 4) 搬出：只搬 curS2 行结果
```

> TND 的叠加式 `actualSeqLengths`（第 3 章 3.3 节）在本阶段接入：每个 batch 的有效 S 都不同，尾块大小逐 batch 计算。

## 4. 尾块对流水的影响

尾块基本块小、单块耗时短，导致 preload 的「掩盖窗口」变窄：

- 正常块能被 2 个 Cube 阶段盖住 1 个长 V1；
- 尾块的 V1 短，但数量多，**收尾清空段（`for PRELOAD_NUM` 循环）占比上升**。

这也是第 3 章 3.5 节 workspace 计算「s2Size 不对齐 64 时 stage 空间从 2 倍升 3 倍」的根源：尾块迫使流水线级数增加、缓冲翻倍。

## 小测验

1. BSND padding 与 TND 尾块，哪种是「必须算的浪费」、哪种是「不算的变长」？
2. 尾块的 V1 里，未初始化列为什么要处理成 -inf/0？不处理会怎样？
3. 为什么尾块越多，preload 流水的收益越差？